<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8"><p style="margin:0 0 8px 0; color:#cba6f7; font-weight:bold; font-size:1.05em;">✅ Session 9 — Error Handling · Solutions</p><p style="margin:0;">Worked, runnable solutions for the 12 <strong>Exercises</strong> and 8 <strong>Code Challenges</strong>. Run top to bottom to verify. Try them in <code>01_errors.ipynb</code> first.</p></div>

### Exercises — Solutions

In [ ]:
from functools import wraps
from contextlib import contextmanager

In [ ]:
# E1 — safe_divide
def safe_divide(a, b):
    try: return a / b
    except ZeroDivisionError: return None

print(safe_divide(6, 2), safe_divide(1, 0))   # 3.0 None

In [ ]:
# E2 — catch (KeyError, IndexError) together
def get(coll, k):
    try: return coll[k]
    except (KeyError, IndexError): return None

print(get({"a": 1}, "a"), get([1, 2], 5))     # 1 None

In [ ]:
# E3 — else runs on success only
def run(s):
    try: n = int(s)
    except ValueError: return "err"
    else: return "ok"

print(run("5"), run("x"))                     # ok err

In [ ]:
# E4 — finally always runs
log = []
def op(fail):
    try:
        if fail: raise ValueError()
    finally:
        log.append("cleanup")
try: op(True)
except ValueError: pass
op(False)
print(log)                                    # ['cleanup', 'cleanup']

In [ ]:
# E5 — custom exception with an attribute
class InsufficientFunds(Exception):
    def __init__(self, amount):
        super().__init__(f"need {amount}")
        self.amount = amount

try: raise InsufficientFunds(50)
except InsufficientFunds as e: print(e, "| amount:", e.amount)   # need 50 | amount: 50

In [ ]:
# E6 — log then re-raise
def logged():
    try: int("x")
    except ValueError:
        print("logging..."); raise        # bare raise re-raises the same exception

try: logged()
except ValueError: print("re-raised OK")

In [ ]:
# E7 — chain with `from`
class ConfigError(Exception): pass
def load(cfg, key):
    try: return cfg[key]
    except KeyError as e: raise ConfigError("missing") from e

try: load({}, "db")
except ConfigError as e: print("cause:", type(e.__cause__).__name__)   # KeyError

In [ ]:
# E8 — my_get via EAFP
def my_get(d, k, default=None):
    try: return d[k]
    except KeyError: return default

print(my_get({"a": 1}, "a"), my_get({"a": 1}, "z", -1))   # 1 -1

In [ ]:
# E9 — validate_record raises on first missing field
class ValidationError(Exception): pass
def validate_record(rec, required):
    for f in required:
        if f not in rec: raise ValidationError(f"missing: {f}")
    return True

try: validate_record({"name": "x"}, ["name", "email"])
except ValidationError as e: print(e)         # missing: email

In [ ]:
# E10 — parse_ints: partial success
def parse_ints(strings):
    values, errors = [], []
    for s in strings:
        try: values.append(int(s))
        except ValueError: errors.append(s)
    return values, errors

print(parse_ints(["1", "x", "3", "y"]))       # ([1, 3], ['x', 'y'])

In [ ]:
# E11 — retry only on a specific exception
def retry_on(fn, exc, times):
    last = None
    for _ in range(times):
        try: return fn()
        except exc as e: last = e            # other exceptions propagate immediately
    raise last

st = {"n": 0}
def flaky():
    st["n"] += 1
    if st["n"] < 2: raise ValueError()
    return "ok"
print(retry_on(flaky, ValueError, 3))         # ok

In [ ]:
# E12 — ensure(cond, exc_type, msg)
def ensure(cond, exc_type, msg):
    if not cond: raise exc_type(msg)

try: ensure(False, ValueError, "bad")
except ValueError as e: print(e)              # bad

### Code Challenges — Solutions

In [ ]:
# C1 — safe_get
def safe_get(seq, i, default=None):
    try: return seq[i]
    except (IndexError, KeyError): return default

print(safe_get([1, 2, 3], 1), safe_get([1, 2], 9, "?"))   # 2 ?

In [ ]:
# C2 — checked_sqrt
def checked_sqrt(x):
    if x < 0: raise ValueError("negative")
    return x ** 0.5

print(checked_sqrt(9))                        # 3.0
try: checked_sqrt(-1)
except ValueError as e: print("raised:", e)

In [ ]:
# C3 — run_all -> (ok, result_or_error) per func
def run_all(funcs):
    out = []
    for f in funcs:
        try: out.append((True, f()))
        except Exception as e: out.append((False, type(e).__name__))
    return out

print(run_all([lambda: 1, lambda: 1/0, lambda: int("x")]))

In [ ]:
# C4 — reraise_as context manager (translate + chain)
class ConfigError(Exception): pass
@contextmanager
def reraise_as(from_exc, to_exc):
    try:
        yield
    except from_exc as e:
        raise to_exc(str(e)) from e

try:
    with reraise_as(KeyError, ConfigError):
        {}["x"]
except ConfigError as e: print("cause:", type(e.__cause__).__name__)   # KeyError

In [ ]:
# C5 — validate_age (type vs value errors)
def validate_age(age):
    if not isinstance(age, int): raise TypeError("age must be int")
    if age < 0: raise ValueError("age must be >= 0")
    return age

print(validate_age(30))                       # 30
try: validate_age(-1)
except ValueError as e: print("raised:", e)

In [ ]:
# C6 — @catch(exc, default) decorator
def catch(exc, default):
    def deco(fn):
        @wraps(fn)
        def w(*a, **k):
            try: return fn(*a, **k)
            except exc: return default
        return w
    return deco

@catch(ZeroDivisionError, float("inf"))
def recip(x): return 1 / x
print(recip(4), recip(0))                     # 0.25 inf

In [ ]:
# C7 — collect all errors, then raise AggregateError
class AggregateError(Exception):
    def __init__(self, errors):
        super().__init__(f"{len(errors)} errors")
        self.errors = errors

def process_all(items, fn):
    results, errors = [], []
    for it in items:
        try: results.append(fn(it))
        except Exception as e: errors.append((it, type(e).__name__))
    if errors: raise AggregateError(errors)
    return results

try: process_all(["1", "x", "3"], int)
except AggregateError as e: print(e, "|", e.errors)   # 1 errors | [('x', 'ValueError')]

In [ ]:
# C8 — parameterized @retry(times, exc)
def retry(times=3, exc=Exception):
    def deco(fn):
        @wraps(fn)
        def w(*a, **k):
            last = None
            for _ in range(times):
                try: return fn(*a, **k)
                except exc as e: last = e
            raise last
        return w
    return deco

st2 = {"n": 0}
@retry(times=5, exc=ValueError)
def f2():
    st2["n"] += 1
    if st2["n"] < 3: raise ValueError()
    return "done"
print(f2(), "after", st2["n"])                # done after 3